# 04 — Evaluation & Results

Compute offline retrieval metrics, validate the two research hypotheses, and produce qualitative recommendation samples for at least five seed titles.

Because the dataset has no user ratings, a **genre-overlap proxy** is used for relevance: two items are relevant if they share at least one genre.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.evaluator import Evaluator

pd.set_option('display.max_colwidth', 60)

In [ ]:
data_dir = PROJECT_ROOT / 'data'
items = pd.read_parquet(data_dir / 'items.parquet')
item_embeddings = np.load(data_dir / 'item_embeddings.npy')
genre_matrix = np.load(data_dir / 'genre_matrix.npy')
print('items      :', items.shape)
print('embeddings :', item_embeddings.shape)
print('genre_matrix:', genre_matrix.shape)

evaluator = Evaluator(items=items, embeddings=item_embeddings, genre_matrix=genre_matrix)

## 6.1 Metrics

| Metric | Meaning |
|---|---|
| `precision@10` | Share of top-10 recs that share a genre with the seed |
| `recall@10` | Share of all genre-relevant items retrieved in top-10 |
| `ndcg@10` | Ranking quality under genre-overlap relevance |
| `coverage` | Fraction of the catalog that appears in any top-10 list |
| `intra_list_diversity` | 1 − mean pairwise cosine similarity inside each rec list |

In [ ]:
metrics = evaluator.compute_all(k=10)
print('Retrieval metrics @10:')
for k, v in metrics.items():
    print(f'  {k:<22s} {v:.4f}')

## 6.2 Hypothesis validation

### Hypothesis 1 — Content-based similarity retrieves on-genre items
If precision@10 is materially higher than the random baseline (genre density of the catalog), H1 is supported.

In [ ]:
# Random-baseline precision = probability that two random items share >=1 genre.
rng = np.random.default_rng(7)
n = len(items)
sample = 1000
a = rng.integers(0, n, size=sample)
b = rng.integers(0, n, size=sample)
random_precision = float(((genre_matrix[a] * genre_matrix[b]).sum(axis=1) > 0).mean())
print(f'Random-pair genre-overlap probability : {random_precision:.4f}')
print(f'Model precision@10                    : {metrics["precision@10"]:.4f}')
print(f'Lift vs random                        : {metrics["precision@10"] / max(random_precision, 1e-6):.2f}x')

### Hypothesis 2 — Recent content is over-represented in rec lists

In [ ]:
temporal = evaluator.temporal_bias(k=10, sample=500)
print('Temporal-bias diagnostics:')
for k, v in temporal.items():
    print(f'  {k:<32s} {v:.4f}')

## 6.3 Sample recommendations

Top-10 recommendations for a diverse set of seed titles. If a seed is missing from the catalog it is automatically skipped.

In [ ]:
def show_recommendations(seed_title: str, top_k: int = 10):
    try:
        recs = evaluator.recommendations_for(seed_title, top_k=top_k)
    except ValueError as exc:
        print(exc)
        return None
    print(f'Seed: {seed_title!r}')
    return recs

# Fall back to any available seeds if the canonical ones are missing.
candidate_seeds = [
    'Stranger Things', 'Breaking Bad', 'The Crown', 'Narcos',
    'Money Heist', 'Dark', 'Black Mirror', 'The Witcher',
]
available = [t for t in candidate_seeds if (items['title'].astype(str).str.lower() == t.lower()).any()]
if len(available) < 5:
    available = available + items['title'].astype(str).head(10).tolist()
seeds = available[:5]
seeds

In [ ]:
for seed in seeds:
    recs = show_recommendations(seed, top_k=10)
    if recs is not None:
        display(recs)